# 실습 6. 상관분석과 선형회귀

사회약학 연구방법 노트의 '실습 6. 상관분석과 선형회귀'에 딸린 노트북입니다. scikit-learn에 들어 있는 당뇨병 환자 442명 자료로 BMI와 1년 뒤 질병 진행 점수의 Pearson·Spearman 상관계수를 구하고(b), 단순회귀(c)와 다중회귀(d)를 statsmodels로 적합한 뒤 논문용 결과표를 만듭니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 6에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 당뇨병 자료를 읽습니다. 자료는 scikit-learn 안에 들어 있어 인터넷 연결 없이 불러옵니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import pandas as pd
import numpy as np
import scipy, statsmodels, matplotlib, sklearn

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("statsmodels ", statsmodels.__version__)
print("matplotlib  ", matplotlib.__version__)
print("scikit-learn", sklearn.__version__)

### 셀 2. 당뇨병 자료 불러오기

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(scaled=False, as_frame=True)
df = data.frame            # 설명변수 10개 + target 1개
print(df.shape)            # (행 수, 열 수)
df.head()

### 셀 3. 자료 설명서에서 변수 정의 확인

In [ ]:
desc = data.DESCR                        # 자료 설명서(문자열)
start = desc.find(":Attribute Information:")
end = desc.find("Source URL")
print(desc[start:end])

### 셀 4. 요약통계와 sex 코딩 확인

In [ ]:
print(df["sex"].value_counts())    # sex에 어떤 값이 몇 명씩 있는지
df["sex"] = df["sex"].astype(int)  # 1.0, 2.0 -> 1, 2 (정수로)
df.describe().T.round(2)           # 변수별 요약통계 (행·열 뒤집기)

열 이름을 틀려 일부러 오류를 내 보는 셀(셀 5)은 이 노트북에서 뺐습니다(사이트의 실습 6 a 절에 있습니다).

### 셀 6. 산점도 행렬

In [ ]:
import matplotlib.pyplot as plt

cols = ["bmi", "bp", "s5", "target"]
pd.plotting.scatter_matrix(df[cols], figsize=(7, 7), alpha=0.35,
                           diagonal="hist", s=12)
plt.suptitle("Scatter matrix: diabetes data (n = 442)")
plt.show()

## b. 상관분석

### 셀 7. Pearson 상관계수, p-value, 95% 신뢰구간

In [ ]:
from scipy import stats

res = stats.pearsonr(df["bmi"], df["target"])   # Pearson 상관
print(res)
print(res.confidence_interval(confidence_level=0.95))

### 셀 8. 손 공식으로 검산하기

In [ ]:
n, r = len(df), res.statistic
t = r * np.sqrt(n - 2) / np.sqrt(1 - r**2)   # 6장 a 절의 t 공식
z, se = np.arctanh(r), 1 / np.sqrt(n - 3)    # Fisher z 변환과 SE
print(f"t = {t:.3f}, df = {n - 2}")
print("95% CI:", np.tanh([z - 1.96 * se, z + 1.96 * se]).round(4))

### 셀 9. Spearman 순위상관계수

In [ ]:
rho = stats.spearmanr(df["bmi"], df["target"])   # Spearman 순위상관
print(rho)

### 셀 10. 상관행렬

In [ ]:
cols = ["age", "bmi", "bp", "s1", "s2", "s3", "s5", "target"]
df[cols].corr().round(2)       # 기본은 Pearson

### 셀 11. 입력 오류 한 건이 상관계수에 주는 영향

In [ ]:
bad = df.copy()               # 원본은 그대로 두고 복사본을 만듦
bad.loc[0, "bmi"] = 321       # 32.1을 소수점 없이 입력한 오류 가정
r_bad = stats.pearsonr(bad["bmi"], bad["target"]).statistic
rho_bad = stats.spearmanr(bad["bmi"], bad["target"]).statistic
print(f"Pearson r    : {res.statistic:.3f} -> {r_bad:.3f}")
print(f"Spearman rho : {rho.statistic:.3f} -> {rho_bad:.3f}")
print("bmi max      :", bad["bmi"].max())

### 셀 12. 입력 오류가 있는 산점도

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].scatter(df["bmi"], df["target"], s=10, alpha=0.5)
ax[0].set_title(f"Original data (r = {res.statistic:.2f})")
ax[1].scatter(bad["bmi"], bad["target"], s=10, alpha=0.5)
ax[1].set_title(f"One data-entry error (r = {r_bad:.2f})")
for a in ax:
    a.set_xlabel("BMI (kg/m2)")
    a.set_ylabel("Disease progression")
plt.tight_layout()
plt.show()

## c. 단순회귀분석

### 셀 13. 단순회귀 적합과 summary()

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols("target ~ bmi", data=df).fit()   # 모형 적합
print(model.summary())                           # 결과 요약표

### 셀 14. 기울기 검정 = 상관 검정 확인

In [ ]:
print(model.pvalues)                     # 계수별 정확한 p-value
print("Pearson p :", res.pvalue)
print("t^2 =", round(model.tvalues["bmi"] ** 2, 2),
      "  F =", round(model.fvalue, 2))
print("r^2 =", round(res.statistic ** 2, 4),
      "  R2 =", round(model.rsquared, 4))
print(model.conf_int().round(3))         # 계수의 95% 신뢰구간

### 셀 15. 잔차 그림 두 가지

In [ ]:
import statsmodels.api as sm

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].scatter(model.fittedvalues, model.resid, s=10, alpha=0.5)
ax[0].axhline(0, color="gray", linestyle="--")
ax[0].set_xlabel("Fitted values")
ax[0].set_ylabel("Residuals")
ax[0].set_title("Residuals vs fitted")
sm.qqplot(model.resid, line="s", ax=ax[1])
ax[1].set_title("Normal Q-Q plot of residuals")
plt.tight_layout()
plt.show()

### 셀 16. 영향력이 큰 환자 찾기

In [ ]:
infl = model.get_influence().summary_frame()    # 환자별 영향력 지표
top = infl.sort_values("cooks_d", ascending=False).head(3)
print(top[["cooks_d", "standard_resid", "hat_diag"]]
      .join(df[["bmi", "target"]]).round(3))
m_wo = smf.ols("target ~ bmi", data=df.drop(index=top.index[0])).fit()
print("slope, all 442 patients:", round(model.params["bmi"], 2))
print("slope, without top 1   :", round(m_wo.params["bmi"], 2))

### 셀 17. 평균의 신뢰구간과 개인의 예측구간

In [ ]:
new = pd.DataFrame({"bmi": [25, 30]})     # 예측하고 싶은 BMI 값
pred = model.get_prediction(new)
pred.summary_frame(alpha=0.05).round(1)

## d. 다중회귀분석

### 셀 18. 다중회귀 적합

In [ ]:
model_adj = smf.ols("target ~ bmi + age + C(sex) + bp + s5",
                    data=df).fit()
print(model_adj.summary())

### 셀 19. 보정 전후 BMI 계수 비교

In [ ]:
b_crude = model.params["bmi"]         # 단순회귀의 bmi 계수
b_adj = model_adj.params["bmi"]       # 다중회귀의 bmi 계수
print(f"bmi coefficient: {b_crude:.2f} -> {b_adj:.2f}")
print(f"change: {(b_adj - b_crude) / b_crude * 100:.1f}%")
print(df[["bmi", "bp", "s5", "target"]].corr().round(2))

### 셀 20. 분산팽창계수(VIF)

In [ ]:
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor)

def vif_table(m):
    X, names = m.model.exog, m.model.exog_names   # 설계행렬, 열 이름
    v = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
    s = pd.Series(v, index=names, name="VIF")    # 이름 붙인 Series
    return s.drop("Intercept").round(2)

vif_table(model_adj)

### 셀 21. 혈청 측정값 6개를 모두 넣으면

In [ ]:
model_all = smf.ols("target ~ bmi + age + C(sex) + bp"
                    " + s1 + s2 + s3 + s4 + s5 + s6", data=df).fit()
print(vif_table(model_all))
print(f"r(s1, s2) = {df['s1'].corr(df['s2']):.2f}")
print("SE of s5:", round(model_adj.bse["s5"], 2), "->",
      round(model_all.bse["s5"], 2))

### 셀 22. 로그 변환한 결과변수

In [ ]:
model_log = smf.ols("np.log(target) ~ bmi + age + C(sex) + bp + s5",
                    data=df).fit()
pct = (np.exp(model_log.params) - 1) * 100          # % 차이
pct_ci = (np.exp(model_log.conf_int()) - 1) * 100   # CI 양 끝 변환
out = pd.concat([model_log.params, pct, pct_ci], axis=1)
out.columns = ["beta (log)", "% diff", "lower %", "upper %"]
out.drop("Intercept").round({"beta (log)": 4, "% diff": 1,
                             "lower %": 1, "upper %": 1})

### 셀 23. 로그 변환이 필요했는지 확인

In [ ]:
sk = lambda x: f"{stats.skew(x):.2f}"      # 왜도를 소수 둘째 자리로
print("skewness of target     :", sk(df["target"]))
print("skewness of log(target):", sk(np.log(df["target"])))
print("skewness of residuals  :", sk(model_adj.resid), "(linear),",
      sk(model_log.resid), "(log)")

### 셀 24. 논문용 결과표 만들기

In [ ]:
def tidy(m):
    ci = m.conf_int()
    f2 = "{:.2f}".format
    est = (m.params.map(f2) + " (" + ci[0].map(f2) + " to "
           + ci[1].map(f2) + ")")
    p = m.pvalues.map(lambda x: "<0.001" if x < 0.001 else f"{x:.3f}")
    return pd.DataFrame({"beta (95% CI)": est, "P": p})

terms = ["bmi", "age", "C(sex)", "bp", "s5"]
crude = pd.concat([tidy(smf.ols(f"target ~ {t}", data=df).fit())
                   .drop("Intercept") for t in terms])
table = crude.join(tidy(model_adj), lsuffix=" crude",
                   rsuffix=" adjusted")
table

## 과제 정답

**과제 1.** 셀 24의 결과표에서 bp 행의 보정 계수(1.04, 95% CI 0.60 to 1.48)를 논문의 Results처럼 한 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. bp의 범위와 표준편차를 확인하고, 10 단위당과 1 표준편차당 계수도 구해 보세요.

### 셀 25. 과제 1 정답. 혈압(bp) 행 읽기

In [ ]:
print(df["bp"].describe().round(1))                # bp의 범위와 표준편차
b = model_adj.params["bp"]                         # 셀 18의 보정 계수
lo, hi = model_adj.conf_int().loc["bp"]
sd_bp = df["bp"].std()
print("per 1 unit :", np.round([b, lo, hi], 2))
print("per 10 units:", np.round(10 * np.array([b, lo, hi]), 1))
print(f"per 1 SD ({sd_bp:.1f}):", np.round(sd_bp * np.array([b, lo, hi]), 1))

**과제 2.** 셀 13의 단순회귀에서 시작해 보정 변수를 하나씩 바꿔 가며 BMI 계수를 다시 구하세요. 나이와 sex만, 여기에 bp만, s5만, bp와 s5를 모두 넣은 모형의 BMI 계수, 95% CI, 표준오차, R²를 한 표로 만들고, 어떤 변수를 넣을 때 계수가 크게 줄어드는지, 왜 그런지 설명하세요.

### 셀 26. 과제 2 정답. 보정 변수를 바꿔 가며 BMI 계수 보기

In [ ]:
sets = {"bmi only": "target ~ bmi",
        "+ age, sex": "target ~ bmi + age + C(sex)",
        "+ age, sex, bp": "target ~ bmi + age + C(sex) + bp",
        "+ age, sex, s5": "target ~ bmi + age + C(sex) + s5",
        "+ age, sex, bp, s5": "target ~ bmi + age + C(sex) + bp + s5"}
rows = []
for name, f in sets.items():
    m = smf.ols(f, data=df).fit()
    lo, hi = m.conf_int().loc["bmi"]               # bmi 계수의 95% CI
    rows.append({"model": name, "bmi": m.params["bmi"], "lower": lo,
                 "upper": hi, "SE": m.bse["bmi"], "R2": m.rsquared})
pd.DataFrame(rows).set_index("model").round(3)

**과제 3.** 동료가 단순회귀의 잔차가 2 표준편차를 넘는 환자를 이상값으로 보고 빼서 다시 적합한 뒤 "이상값 13명을 제외하자 모형의 설명력이 R² 0.34에서 0.41로 좋아졌고, BMI 1 kg/m²당 진행 점수가 10.8점(95% CI 9.6–12.1) 높았다"라고 썼습니다. 동료의 분석을 재현하고, 무엇이 왜 틀렸는지 두 가지 이상 설명하세요. 셀 16의 영향력 확인과 비교해 보세요.

### 셀 27. 과제 3 정답. 잔차가 큰 환자를 뺀 분석

In [ ]:
r = model.resid                                  # 셀 13 단순회귀의 잔차
keep = r.abs() <= 2 * r.std()                    # 동료: 잔차가 2 SD 넘는 환자를 뺌
m_trim = smf.ols("target ~ bmi", data=df[keep]).fit()
print("removed:", (~keep).sum(), "patients")
print(df.loc[~keep, ["bmi", "target"]].describe().loc[["min", "max"]])
for name, m in [("all 442", model), ("trimmed", m_trim)]:
    lo, hi = m.conf_int().loc["bmi"]
    print(f"{name:8s} slope {m.params['bmi']:.2f} ({lo:.2f} to {hi:.2f})"
          f"  R2 {m.rsquared:.3f}")